# Databricks/PySpark dla Data Analyst/Engineer — Moduł 11: Architektura Lakehouse

Ten moduł łączy wszystko, co poznaliśmy dotychczas (DataFrame API, Spark SQL, Delta
Lake), w jeden spójny obraz: jak wygląda NOWOCZESNA architektura danych w firmie
korzystającej z Databricks, i dlaczego określa się ją mianem **lakehouse**.

## Spis treści
1. [Data lake kontra data warehouse kontra lakehouse](#sec1)
2. [Medallion Architecture: warstwy bronze/silver/gold](#sec2)
3. [Budujemy mini-lakehouse na naszych danych](#sec3)
4. [Unity Catalog: podstawy](#sec4)
5. [Podsumowanie i ćwiczenia](#sec5)


Uruchamiamy (albo pobieramy już istniejącą) `SparkSession` — punkt wejścia do wszystkiego w Sparku. W Databricks `spark` jest już gotowe automatycznie w każdym notebooku; ten sam kod działa też lokalnie.

In [ ]:
from pyspark.sql import SparkSession

# Dodatkowe opcje JVM (--add-opens) sa potrzebne, gdy uruchamiasz Sparka lokalnie na
# Javie 17+ i chcesz uzywac pandas UDF / Apache Arrow (Modul 8) -- nowsze JDK domyslnie
# blokuja refleksyjny dostep do pamieci poza stosem, ktorego potrzebuje Arrow. W
# Databricks nie trzeba tego ustawiac recznie -- platforma robi to za Ciebie.
_opcje_jvm = (
    "--add-opens=java.base/java.nio=ALL-UNNAMED "
    "--add-opens=java.base/java.lang=ALL-UNNAMED "
    "--add-opens=java.base/sun.nio.ch=ALL-UNNAMED"
)

spark = (
    SparkSession.builder.appName("kurs_spark")
    .config("spark.driver.extraJavaOptions", _opcje_jvm)
    .config("spark.executor.extraJavaOptions", _opcje_jvm)
    .getOrCreate()
)
spark


Wczytujemy nasz wspólny zestaw danych — dane sklepu internetowego (`klienci`, `produkty`, `pracownicy`, `zamowienia`, `pozycje_zamowien`), dokładnie ten sam schemat co w kursie SQL, ale tym razem jako DataFrame'y Sparka.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(7)

# --- klienci -----------------------------------------------------------
n_klienci = 300
miasta = ["Warszawa", "Krakow", "Gdansk", "Wroclaw", "Poznan"]
segmenty = ["Nowy", "Standardowy", "Premium"]
prawdop_segmentu = [0.35, 0.45, 0.20]

klienci_pd = pd.DataFrame({
    "klient_id": np.arange(1, n_klienci + 1),
    "miasto": rng.choice(miasta, size=n_klienci, p=[0.30, 0.25, 0.15, 0.15, 0.15]),
    "segment": rng.choice(segmenty, size=n_klienci, p=prawdop_segmentu),
    "dni_od_rejestracji": rng.integers(1, 900, size=n_klienci),
})

# --- produkty ------------------------------------------------------------
n_produkty = 40
kategorie = ["Elektronika", "Odziez", "Dom", "Sport", "Ksiazki"]
srednie_ceny = {"Elektronika": 1200, "Odziez": 150, "Dom": 250, "Sport": 300, "Ksiazki": 50}

kategorie_produktow = rng.choice(kategorie, size=n_produkty)
produkty_pd = pd.DataFrame({
    "produkt_id": np.arange(1, n_produkty + 1),
    "kategoria": kategorie_produktow,
    "cena": [
        round(max(5.0, rng.normal(srednie_ceny[k], srednie_ceny[k] * 0.3)), 2)
        for k in kategorie_produktow
    ],
})

# --- pracownicy ----------------------------------------------------------
n_pracownicy = 12
regiony = ["Polnoc", "Poludnie", "Wschod", "Zachod"]
pracownicy_pd = pd.DataFrame({
    "pracownik_id": np.arange(1, n_pracownicy + 1),
    "region": rng.choice(regiony, size=n_pracownicy),
    "dzial": rng.choice(["Sprzedaz", "Obsluga"], size=n_pracownicy, p=[0.6, 0.4]),
})

# --- zamowienia ------------------------------------------------------------
n_zamowienia = 3000
klient_id_zamowien = rng.choice(klienci_pd["klient_id"], size=n_zamowienia)
pracownik_id_zamowien = rng.choice(pracownicy_pd["pracownik_id"], size=n_zamowienia)

segment_mapa = klienci_pd.set_index("klient_id")["segment"]
segment_zamowien = klient_id_zamowien.astype(object)
segment_zamowien = pd.Series(klient_id_zamowien).map(segment_mapa).values

# prawdopodobienstwo anulowania zalezy od segmentu klienta (celowo wbudowany efekt)
prawdop_anulowania = np.where(
    segment_zamowien == "Nowy", 0.22,
    np.where(segment_zamowien == "Standardowy", 0.10, 0.04),
)
anulowane = rng.binomial(1, prawdop_anulowania)
status_zamowien = np.where(
    anulowane == 1, "Anulowane",
    rng.choice(["Zlozone", "Wyslane", "Dostarczone"], size=n_zamowienia, p=[0.2, 0.3, 0.5]),
)

data_bazowa = pd.Timestamp("2024-01-01")
dni_offset = rng.integers(0, 730, size=n_zamowienia)

zamowienia_pd = pd.DataFrame({
    "zamowienie_id": np.arange(1, n_zamowienia + 1),
    "klient_id": klient_id_zamowien,
    "pracownik_id": pracownik_id_zamowien,
    "data_zamowienia": [data_bazowa + pd.Timedelta(days=int(d)) for d in dni_offset],
    "status": status_zamowien,
})

# --- pozycje_zamowien ------------------------------------------------------
wiersze_pozycji = []
pozycja_id = 1
cena_produktu = produkty_pd.set_index("produkt_id")["cena"]

for zamowienie_id in zamowienia_pd["zamowienie_id"]:
    liczba_pozycji = rng.integers(1, 5)
    wybrane_produkty = rng.choice(produkty_pd["produkt_id"], size=liczba_pozycji, replace=False)
    for produkt_id in wybrane_produkty:
        ilosc = int(rng.integers(1, 4))
        rabat = rng.choice([0.0, 0.05, 0.10, 0.15], p=[0.55, 0.20, 0.15, 0.10])
        wiersze_pozycji.append({
            "pozycja_id": pozycja_id,
            "zamowienie_id": int(zamowienie_id),
            "produkt_id": int(produkt_id),
            "ilosc": ilosc,
            "cena_jednostkowa": float(cena_produktu[produkt_id]),
            "rabat": float(rabat),
        })
        pozycja_id += 1

pozycje_zamowien_pd = pd.DataFrame(wiersze_pozycji)

# --- konwersja na DataFrame'y Sparka ---------------------------------------
klienci = spark.createDataFrame(klienci_pd)
produkty = spark.createDataFrame(produkty_pd)
pracownicy = spark.createDataFrame(pracownicy_pd)
zamowienia = spark.createDataFrame(zamowienia_pd)
pozycje_zamowien = spark.createDataFrame(pozycje_zamowien_pd)

klienci.show(5)


<a id="sec1"></a>
## 1. Data lake kontra data warehouse kontra lakehouse

Historycznie firmy budowały dwa OSOBNE systemy do przechowywania danych, każdy z innymi
kompromisami:

- **Data warehouse** (hurtownia danych, np. tradycyjne bazy analityczne) -- dane
  ustrukturyzowane, ze SCHEMATEM narzuconym z góry, zoptymalizowane pod SQL i raporty
  BI. Mocne strony: szybkie zapytania, silne gwarancje spójności (ACID). Słabości:
  drogie przy bardzo dużej skali, słabo radzi sobie z danymi nieustrukturyzowanymi
  (obrazy, logi, JSON o zmiennym kształcie), zwykle zamknięty ekosystem.
- **Data lake** (jezioro danych, np. surowe pliki na S3/ADLS/HDFS) -- WSZYSTKIE typy
  danych, tanie przechowywanie na dużą skalę, elastyczny schemat ("schema-on-read").
  Słabości: brak ACID (dokładnie problemy z Modułu 10 -- niespójne zapisy, brak
  egzekwowania schematu), łatwo zamienia się w tzw. "data swamp" (bagno danych) bez
  ładu i porządku.
- **Lakehouse** -- Delta Lake (Moduł 10) DOKŁADA do taniego, elastycznego magazynu typu
  data lake własności znane z hurtowni: transakcje ACID, egzekwowanie schematu, szybkie
  zapytania SQL, kontrolę dostępu. Rezultat: JEDNO miejsce przechowywania danych dla
  raportów BI, analiz ad-hoc, inżynierii danych I uczenia maszynowego (Moduły 12-13) --
  zamiast utrzymywania osobnych kopii danych w dwóch różnych systemach.

> ⚡ **Dlaczego to ważne w praktyce, nie tylko teoretycznie**
>
> Przed lakehouse, typowa firma kopiowała te same dane DWA RAZY: raz do data lake (dla data science/ML), raz do hurtowni (dla BI/raportów) -- z osobnymi potokami ETL do utrzymania, opóźnieniami w synchronizacji i podwójnymi kosztami przechowywania. Lakehouse (Databricks + Delta Lake) eliminuje tę duplikację: JEDNO źródło prawdy, czytane zarówno przez notatniki Pythona/Sparka, jak i narzędzia BI przez zwykłe zapytania SQL.

<a id="sec2"></a>
## 2. Medallion Architecture: warstwy bronze/silver/gold

**Medallion Architecture** to standardowy wzorzec organizacji danych w lakehouse --
dane przepływają przez TRZY kolejne warstwy, każda o coraz wyższej jakości i coraz
bardziej gotowa do analizy biznesowej:

- **Bronze (surowa)** -- dane w DOKŁADNIE takiej postaci, w jakiej przyszły ze źródła
  (pliki CSV, zdarzenia z API, eksporty z innego systemu). Zapisywane w całości, bez
  czyszczenia -- to Twoja "kopia zapasowa prawdy", do której zawsze można wrócić.
- **Silver (oczyszczona)** -- dane po CZYSZCZENIU: usunięte duplikaty, poprawione typy,
  odfiltrowane błędne rekordy, połączone z innymi tabelami. Wciąż na poziomie
  szczegółowym (pojedyncze wiersze/zdarzenia), ale już wiarygodne.
- **Gold (biznesowa)** -- dane ZAGREGOWANE do postaci gotowej dla konkretnych
  odbiorców biznesowych: dzienne podsumowania sprzedaży, metryki per klient/region,
  tabele bezpośrednio zasilające dashboardy BI.

> ⚠️ **Każda warstwa to OSOBNA tabela, nie krok pośredni znikający po przetworzeniu**
>
> Kluczowa różnica względem tradycyjnego ETL: bronze/silver/gold to TRWAŁE tabele, każda dostępna do odczytu niezależnie. Dzięki temu, jeśli logika czyszczenia w warstwie silver okaże się błędna, możesz ją naprawić i PRZELICZYĆ silver na nowo z bronze -- bez utraty żadnych danych źródłowych, bo bronze wciąż leży nietknięte.

<a id="sec3"></a>
## 3. Budujemy mini-lakehouse na naszych danych

Zbudujmy uproszczony potok bronze -> silver -> gold na naszym zestawie danych
sklepu internetowego. Dla przejrzystości ćwiczenia używamy zwykłego Parquetu (jak w
Module 6) -- w PRAWDZIWYM Databricks każda z tych warstw byłaby tabelą Delta (Moduł 10),
żeby korzystać z ACID, time travel i MERGE INTO na każdym etapie.

In [ ]:
import os

os.makedirs("lakehouse", exist_ok=True)

# --- BRONZE: dane dokladnie takie, jak przyszly ze zrodla, bez zadnych zmian ---
zamowienia.write.mode("overwrite").parquet("lakehouse/bronze_zamowienia")
pozycje_zamowien.write.mode("overwrite").parquet("lakehouse/bronze_pozycje_zamowien")
klienci.write.mode("overwrite").parquet("lakehouse/bronze_klienci")

print("Warstwa bronze zapisana.")


In [ ]:
# --- SILVER: czyszczenie -- odrzucamy zamowienia anulowane (nieprzydatne biznesowo
# do analizy sprzedazy), laczymy z klientami, dodajemy wartosc pozycji ---
from pyspark.sql import functions as F

bronze_zamowienia = spark.read.parquet("lakehouse/bronze_zamowienia")
bronze_pozycje = spark.read.parquet("lakehouse/bronze_pozycje_zamowien")
bronze_klienci = spark.read.parquet("lakehouse/bronze_klienci")

silver_zamowienia = (
    bronze_zamowienia.filter(F.col("status") != "Anulowane")
    .join(bronze_klienci, "klient_id")
    .join(bronze_pozycje, "zamowienie_id")
    .withColumn(
        "wartosc_pozycji",
        F.col("ilosc") * F.col("cena_jednostkowa") * (1 - F.col("rabat")),
    )
)

silver_zamowienia.write.mode("overwrite").parquet("lakehouse/silver_zamowienia")
print(f"Warstwa silver: {silver_zamowienia.count()} wierszy (po odrzuceniu anulowanych).")


In [ ]:
# --- GOLD: agregacja gotowa dla biznesu -- dzienna sprzedaz per miasto ---
silver = spark.read.parquet("lakehouse/silver_zamowienia")

gold_sprzedaz_per_miasto = silver.groupBy("miasto", "data_zamowienia").agg(
    F.round(F.sum("wartosc_pozycji"), 2).alias("przychod"),
    F.countDistinct("zamowienie_id").alias("liczba_zamowien"),
)

gold_sprzedaz_per_miasto.write.mode("overwrite").parquet("lakehouse/gold_sprzedaz_per_miasto")
gold_sprzedaz_per_miasto.orderBy(F.col("przychod").desc()).show(10)


> 🧱 **To samo w Databricks -- z Delta Lake na każdym etapie**
>
> W realnym środowisku Databricks każdy z trzech powyższych zapisów użyłby `.write.format("delta")` zamiast `.write.parquet(...)`, a przeliczanie silver z bronze często odbywałoby się przez `MERGE INTO` (Moduł 10) zamiast pełnego `overwrite` -- tak, żeby doliczać tylko NOWE/zmienione zamówienia, zamiast przeliczać całą historię za każdym razem. Jobs/Workflows w Databricks (Moduł 14) harmonogramują tego typu potoki np. co godzinę.

<a id="sec4"></a>
## 4. Unity Catalog: podstawy

**Unity Catalog** to system zarządzania metadanymi i uprawnieniami w Databricks --
rozwiązuje pytanie "jakie tabele mamy, kto może je czytać/zmieniać, i skąd pochodzą
dane". Kluczowe pojęcia:

- **Trójpoziomowa przestrzeń nazw**: `katalog.schemat.tabela` (np.
  `sklep.gold.sprzedaz_per_miasto`) -- dodatkowy poziom (`katalog`) ponad znaną z SQL-a
  strukturą `schemat.tabela` (Moduł 8 kursu SQL), przydatny do rozdzielenia np.
  środowisk `produkcja`/`deweloperskie` albo różnych zespołów.
- **Uprawnienia na poziomie kolumn i wierszy** -- można nadać dostęp do CAŁEJ tabeli
  `gold.sprzedaz_per_miasto`, ale zablokować dostęp do konkretnej wrażliwej kolumny
  (np. dane osobowe) dla wybranych użytkowników.
- **Lineage (pochodzenie danych)** -- Unity Catalog automatycznie śledzi, z JAKICH
  tabel powstała dana tabela (np. że `gold.sprzedaz_per_miasto` pochodzi z
  `silver.zamowienia`, która pochodzi z `bronze.zamowienia`) -- bezcenne przy
  debugowaniu "skąd wzięła się ta dziwna liczba w raporcie".

> ⚡ **spark.catalog z Modułu 5 to lokalny, uproszczony odpowiednik**
>
> `spark.catalog.listTables()` z Modułu 5 pokazuje tymczasowe widoki w BIEŻĄCEJ sesji -- znikają, gdy notatnik się zamknie. Unity Catalog robi to samo, ale na poziomie CAŁEJ organizacji: trwałe tabele widoczne dla wszystkich uprawnionych użytkowników, we wszystkich notatnikach i klastrach, z pełną historią uprawnień i dostępu.

<a id="sec5"></a>
## 5. Podsumowanie i ćwiczenia

W tym module poznaliśmy:
- różnice między data lake, data warehouse i lakehouse -- i dlaczego lakehouse eliminuje potrzebę dwóch osobnych systemów,
- Medallion Architecture: warstwy bronze (surowa) / silver (oczyszczona) / gold (biznesowa),
- praktyczne zbudowanie mini-potoku bronze -> silver -> gold na naszych danych,
- Unity Catalog: trójpoziomowa przestrzeń nazw, uprawnienia, lineage danych.

### 📝 Ćwiczenie 1: Warstwa bronze dla pracowników

Zapisz `pracownicy` jako warstwę bronze (`lakehouse/bronze_pracownicy`, Parquet, bez żadnych zmian).

<details>
<summary>Pokaż rozwiązanie</summary>

```python
pracownicy.write.mode("overwrite").parquet("lakehouse/bronze_pracownicy")
print("Zapisano bronze_pracownicy.")
```

</details>

### 📝 Ćwiczenie 2: Warstwa silver -- czyszczenie produktów

Zbuduj warstwę silver dla `produkty`: odfiltruj produkty z ceną poniżej 5 (uznane za błędne dane), zapisz jako `lakehouse/silver_produkty`.

<details>
<summary>Pokaż rozwiązanie</summary>

```python
produkty.write.mode("overwrite").parquet("lakehouse/bronze_produkty")

bronze_produkty = spark.read.parquet("lakehouse/bronze_produkty")
silver_produkty = bronze_produkty.filter(F.col("cena") >= 5)
silver_produkty.write.mode("overwrite").parquet("lakehouse/silver_produkty")
print(f"Silver: {silver_produkty.count()} z {bronze_produkty.count()} produktow.")
```

</details>

### 📝 Ćwiczenie 3: Warstwa gold -- przychód per kategoria

Na podstawie `lakehouse/silver_zamowienia` z sekcji 3, zbuduj warstwę gold z sumą przychodu per `kategoria` produktu (wymaga połączenia z `produkty`).

<details>
<summary>Pokaż rozwiązanie</summary>

```python
silver = spark.read.parquet("lakehouse/silver_zamowienia")

gold_per_kategoria = silver.join(produkty, "produkt_id").groupBy("kategoria").agg(
    F.round(F.sum("wartosc_pozycji"), 2).alias("przychod")
).orderBy(F.col("przychod").desc())

gold_per_kategoria.write.mode("overwrite").parquet("lakehouse/gold_przychod_per_kategoria")
gold_per_kategoria.show()
```

</details>

### 📝 Ćwiczenie 4: Symulacja trójpoziomowej przestrzeni nazw

Bez prawdziwego Unity Catalog, zasymuluj nazewnictwo `katalog.schemat.tabela` przez zapis do folderu `lakehouse/sklep/gold/sprzedaz_per_miasto` zamiast płaskiej nazwy -- pokazuje to samą IDEĘ hierarchii, nawet bez pełnej funkcjonalności katalogu.

<details>
<summary>Pokaż rozwiązanie</summary>

```python
gold_sprzedaz_per_miasto = spark.read.parquet("lakehouse/gold_sprzedaz_per_miasto")
gold_sprzedaz_per_miasto.write.mode("overwrite").parquet(
    "lakehouse/sklep/gold/sprzedaz_per_miasto"
)
print("Zapisano pod hierarchiczna sciezka imitujaca katalog.schemat.tabela.")
```

</details>

> 🔥 **Wyzwanie: pełny potok z lineage w komentarzach**
>
> Zbuduj kompletny potok bronze -> silver -> gold dla ANALIZY WYDAJNOŚCI PRACOWNIKÓW: bronze (`zamowienia`, `pracownicy` -- surowe), silver (połączone, odfiltrowane do statusu `'Dostarczone'`), gold (suma wartości zamówień per pracownik + region, posortowana malejąco). Udokumentuj w komentarzach kodu pochodzenie KAŻDEJ tabeli (z jakiej warstwy/tabeli powstała) -- ręczna symulacja tego, co Unity Catalog śledzi automatycznie.

<details>
<summary>Pokaż rozwiązanie</summary>

```python
# BRONZE: zamowienia i pracownicy w postaci surowej
zamowienia.write.mode("overwrite").parquet("lakehouse/bronze_zamowienia_v2")
pracownicy.write.mode("overwrite").parquet("lakehouse/bronze_pracownicy_v2")

# SILVER: pochodzi z bronze_zamowienia_v2 + bronze_pracownicy_v2 + pozycje_zamowien
# (surowe, nie przez lakehouse -- w realnym potoku tez bylyby to osobne bronze)
bronze_zam = spark.read.parquet("lakehouse/bronze_zamowienia_v2")
bronze_prac = spark.read.parquet("lakehouse/bronze_pracownicy_v2")

silver_wydajnosc = (
    bronze_zam.filter(F.col("status") == "Dostarczone")
    .join(bronze_prac, "pracownik_id")
    .join(pozycje_zamowien, "zamowienie_id")
    .withColumn(
        "wartosc_pozycji",
        F.col("ilosc") * F.col("cena_jednostkowa") * (1 - F.col("rabat")),
    )
)
silver_wydajnosc.write.mode("overwrite").parquet("lakehouse/silver_wydajnosc_pracownikow")

# GOLD: pochodzi z silver_wydajnosc_pracownikow
silver = spark.read.parquet("lakehouse/silver_wydajnosc_pracownikow")
gold_wydajnosc = silver.groupBy("pracownik_id", "region").agg(
    F.round(F.sum("wartosc_pozycji"), 2).alias("suma_wartosci")
).orderBy(F.col("suma_wartosci").desc())

gold_wydajnosc.write.mode("overwrite").parquet("lakehouse/gold_wydajnosc_pracownikow")
gold_wydajnosc.show(10)
```

W prawdziwym Unity Catalog to pochodzenie (ktora tabela powstala z ktorej) bylby widoczne automatycznie w interfejsie graficznym jako diagram lineage -- tutaj musimy je opisac recznie w komentarzach.

</details>

## Co dalej?

W **Module 12** wracamy do w pełni wykonywalnego kodu -- zaczynamy **MLlib**, moduł
uczenia maszynowego Sparka: `Pipeline`, `VectorAssembler`, `StringIndexer` i pierwsze
modele regresji rozproszonej.